In [ ]:
!pip install -U transformers datasets peft trl accelerate

In [ ]:
%pip install --upgrade torchao peft

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    torch_dtype="auto",
    device_map="auto"
)

from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.1,
    task_type="CAUSAL_LM"
)



In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
from transformers import DataCollatorForLanguageModeling
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("tatsu-lab/alpaca", split="train")
raw_dataset = raw_dataset.shuffle(seed=42).select(range(1000)) # take a small sample

print(raw_dataset)

In [ ]:
def preprocess_function(examples):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        if inp.strip():
            full_prompt = f"Instruction:\n{inst}\nInput:\n{inp}\n\nResponse:\n{out}{tokenizer.eos_token}"
        else:
            full_prompt = f"Instruction:\n{inst}\n\nResponse:\n{out}{tokenizer.eos_token}"
        texts.append(full_prompt)

    model_inputs = tokenizer(
        texts,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

tokenized_dataset = raw_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_dataset.column_names
)

In [ ]:
print(tokenized_dataset[0].keys())

In [ ]:
import torch
model.eval()

def chat_with_base(prompt):
    full_prompt = f"Instruction:\n{prompt}\n\nResponse:\n"

    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id
        )


    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("Response:\n")[-1].strip()

questions = [
    "谁是世界上最聪明的动物？",
    "给我讲一个关于程序员的笑话。",
    "已知 1+1=2，那么 2+2 等于几？"
]

for q in questions:
    print(f"❓ 问题: {q}")
    print(f"🤖 回答: {chat_with_base(q)}")
    print("-" * 30)

In [ ]:
model.train()

sft_args = TrainingArguments(
    output_dir="./qwen-sft-lora-alpaca",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=sft_args,
    data_collator=data_collator
)
trainer.train()

In [ ]:
import textwrap

model.eval()

test_questions = [
    "玉皇大帝住在平流层还是对流层？",
]


for i, q in enumerate(test_questions):
    wrapped_q = textwrap.fill(f"Q: {q}", width=40)
    print(f" {wrapped_q}")


    answer = chat_after_train(q)
    wrapped_a = textwrap.fill(answer, width=40)
    print(f"\n A：\n{wrapped_a}")


In [ ]:
model.train()

sft_args_v2 = TrainingArguments(
    output_dir="./qwen-sft-lora-alpaca",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)


trainer_v2 = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=sft_args_v2,
    data_collator=data_collator,
)
trainer.train()

In [ ]:
import textwrap

model.eval()

test_questions = [
    "玉皇大帝住在平流层还是对流层？",
]


for i, q in enumerate(test_questions):
    wrapped_q = textwrap.fill(f"Q: {q}", width=40)
    print(f" {wrapped_q}")


    answer = chat_after_train(q)
    wrapped_a = textwrap.fill(answer, width=40)
    print(f"\n A：\n{wrapped_a}")


In [ ]:
model.train()

sft_args = TrainingArguments(
    output_dir="./qwen-sft-lora-alpaca",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=sft_args,
    data_collator=data_collator
)
trainer.train()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

steps = list(range(1, 64))
loss = [2.091744,1.950227,1.659984,1.668308,1.554229,1.805194,1.530406,1.963491,1.386885,1.741460,1.737808,1.535838,1.693873,1.591856,1.447770,1.605993,1.491091,1.616001,1.678968,1.836159,1.452732,1.510481,1.983282,1.607842,1.597297,1.736047,1.875518,1.734338,1.478117,1.670699,1.561974,1.659898,2.060782,1.696914,1.680250,1.661918,1.734064,1.587220,1.539779,1.233748,1.421810,1.622351,1.551322,1.508234,1.813173,1.734267,1.523119,1.561761,1.639399,1.699475,1.875740,1.638129,1.393421,1.381796,1.433730,1.555849,1.557959,1.788262,1.505097,1.481533,1.884855,1.636372,1.422961]

def moving_avg(data, w=7):
    return [np.mean(data[max(0,i-w//2):min(len(data),i+w//2+1)]) for i in range(len(data))]

avg = moving_avg(loss)

plt.figure(figsize=(10, 4))
plt.plot(steps, loss, color='#5DCAA5', linewidth=1.2, alpha=0.6, label='Training loss')
plt.plot(steps, avg, color='#888780', linewidth=2, linestyle='--', label='Moving avg (w=7)')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Qwen2.5-0.5B Fine-tuning — Training Loss 1')
plt.legend()
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print(f"Loss std: {np.std(loss):.4f}")
print(f"Final avg loss: {np.mean(loss[-10:]):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

steps = list(range(1, 64))
loss = [2.091744,2.210814,1.904348,1.893641,1.643988,1.898003,1.575313,1.996142,1.404533,1.758640,1.747597,1.540480,1.701246,1.594489,1.450149,1.624086,1.501287,1.628978,1.694276,1.825665,1.453746,1.512874,1.983642,1.608111,1.596676,1.737663,1.878234,1.736624,1.479691,1.669910,1.564108,1.661495,2.063033,1.699638,1.678369,1.665829,1.735157,1.585802,1.543383,1.231353,1.421491,1.619578,1.550439,1.511143,1.808790,1.730526,1.528189,1.564099,1.642812,1.702275,1.873742,1.641191,1.398743,1.381959,1.435417,1.560508,1.557296,1.788552,1.505751,1.491855,1.884820,1.640473,1.429327]

def moving_avg(data, w=7):
    return [np.mean(data[max(0,i-w//2):min(len(data),i+w//2+1)]) for i in range(len(data))]

avg = moving_avg(loss)

plt.figure(figsize=(10, 4))
plt.plot(steps, loss, color='#5DCAA5', linewidth=1.2, alpha=0.5, label='Training loss')
plt.plot(steps, avg, color='#888780', linewidth=2, linestyle='--', label='Moving avg (w=7)')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Qwen2.5-0.5B Fine-tuning — Training Loss 2')
plt.legend()
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print(f"Loss std: {np.std(loss):.4f}")
print(f"Final avg loss: {np.mean(loss[-10:]):.4f}")